# Assignment 2 - Advanced Temporal Encoding, Clustering, and Model Comparison

This notebook extends the existing leakage-safe forecasting pipeline. It keeps the original direct, hierarchical, and baseline workflow intact, then adds cyclical temporal features, bus load-shape clustering, clustering-enhanced model variants, and comparative analysis.

## Why Cyclical Encoding

Raw integer time fields imply false distances: HE24 looks far from HE1, Sunday looks far from Monday, and December looks far from January. Sine/cosine encodings place periodic values on a circle, so adjacent operational periods remain close in feature space. This is especially important for electrical demand because daily routines, weekday/weekend behavior, and seasonal weather-driven cycles repeat.

## Leakage Safety

The advanced workflow preserves the existing leakage controls: chronological folds only, shifted rolling features, no centered windows, validation historical averages fitted from the training window, lag masking with `forecast_created_at`, and no random train/test split. Clustering is fit only on training-window bus load shapes, then labels are attached to validation rows.

In [ ]:
from pathlib import Path
import sys
import pandas as pd

repo_root = Path.cwd()
for candidate in [repo_root, *repo_root.parents]:
    if (candidate / "data").exists() and (candidate / "solution" / "assignment_2" / "src").exists():
        repo_root = candidate
        break
else:
    raise FileNotFoundError("Could not find ECESIS repository root from notebook path.")

src_dir = repo_root / "solution" / "assignment_2" / "src"
outputs_dir = repo_root / "solution" / "assignment_2" / "outputs"
outputs_dir.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(src_dir))

pd.set_option("display.max_columns", None)


## Run Advanced Prototype

The prototype uses the same bounded `2022 Jan-Mar -> 2022 Apr` fold and deterministic high-volume bus subset. Clustering and model comparison are configurable so the workflow can scale later without changing the core logic.

In [ ]:
from advanced_experiments import run_advanced_experiments

advanced = run_advanced_experiments(outputs_dir, n_buses=20, n_clusters=4, use_gmm=True)
{k: v.shape for k, v in advanced.items()}

## Load-Shape Clustering

Buses are represented by normalized 24-hour average load profiles. Normalization preserves shape rather than scale, so a small bus and a large bus with similar daily structure can share an archetype. KMeans provides hard demand-archetype assignments. GMM provides probabilistic assignments and can represent overlapping operational patterns, although small prototype samples may produce overconfident probabilities.

In [ ]:
clusters = pd.read_csv(outputs_dir / "bus_clusters.csv")
cluster_stats = pd.read_csv(outputs_dir / "bus_cluster_stats.csv")
cluster_diagnostics = pd.read_csv(outputs_dir / "bus_cluster_diagnostics.csv")
clusters.head(), cluster_stats, cluster_diagnostics

## Model Experiments

The advanced comparison includes raw-time HistGradientBoosting, cyclical HistGradientBoosting, cyclical XGBoost, XGBoost with KMeans cluster features, hierarchical zone allocation, and baselines. CatBoost is availability-aware; in the current environment it runs successfully and is included in the comparison table.

In [ ]:
experiment_log = pd.read_csv(outputs_dir / "advanced_model_experiment_log.csv")
advanced_eval = pd.read_csv(outputs_dir / "advanced_evaluation_summary.csv")
experiment_log

In [ ]:
advanced_eval.sort_values(["horizon", "level", "wmape"])

## Comparative Interpretation

Direct bus forecasting captures local bus detail but is noisier. Zone forecasting is smoother because aggregation cancels some bus-level volatility, so hierarchical allocation may generalize better when bus histories are sparse or unstable. Clustering helps transfer information across buses with similar operating shapes, giving the direct model a structural context beyond bus ID and zone ID. Boosting models are well suited here because they learn nonlinear interactions among calendar cycles, lagged demand, rolling history, and categorical context.

In [ ]:
comparative = pd.read_csv(outputs_dir / "advanced_comparative_summary.csv")
comparative

## Visualization Outputs

Generated plots include representative KMeans cluster centers, GMM cluster distribution, actual-vs-predicted curves, residual distributions, model WMAPE comparison, XGBoost feature importance, and a lightweight SHAP summary for the XGBoost + KMeans variant.

In [ ]:
plot_files = sorted(p.name for p in outputs_dir.glob("*.png") if any(token in p.name for token in ["cluster", "predicted", "residual", "wmape", "feature_importance"]))
plot_files

## Reporting Summary

The forecasting problem has strong temporal periodicity and hierarchical spatial structure. Cyclical encoding helps models learn periodic demand behavior. Clustering is a structural feature engineering method, not a standalone forecast. KMeans gives hard archetypes; GMM allows softer grouping. The final study compares direct versus hierarchical forecasting, boosting methods, clustering-enhanced forecasting, and temporal encoding choices, with attention to accuracy, stability, interpretability, and computational scalability.